In [0]:
%sql
CREATE OR REPLACE TABLE dev_credit_risk.gold.fact_carteira_credito AS

WITH pagamentos_por_contrato AS (
    SELECT
        id_contrato,

        COUNT(*) AS qtd_pagamentos,

        SUM(valor_pago) AS valor_total_pago,

        AVG(
        CASE
                WHEN dias_atraso <> 999 THEN dias_atraso
                ELSE NULL
            END
        ) AS media_dias_atraso,

        MAX(
        CASE
                WHEN dias_atraso <> 999 THEN dias_atraso
                ELSE NULL
            END
        ) AS max_dias_atraso,

        SUM(
        CASE
                WHEN dias_atraso <> 999
                AND dias_atraso > 0 THEN 1
                ELSE 0
            END
        ) AS qtd_pagamentos_atraso,

        SUM(
        CASE
                WHEN dias_atraso <> 999
                AND dias_atraso >= 90 THEN 1
                ELSE 0
            END
        ) AS qtd_pagamentos_default_90,

        SUM(
            CASE
                WHEN dias_atraso > 0 THEN valor_pago
                ELSE 0
            END
        ) AS valor_pago_em_atraso,

        MIN(dt_vencimento) AS primeira_data_vencimento,

        MAX(dt_pagamento) AS ultima_data_pagamento,

        SUM(
            CASE
                WHEN dias_atraso = 999 THEN 1
                ELSE 0
            END
        ) AS qtd_pagamentos_inconsistentes

    FROM dev_credit_risk.silver.fact_pagamento
    GROUP BY
        id_contrato
)

SELECT
    fc.id_contrato,
    fc.id_cliente,
    fc.id_produto,
    fc.id_score,

    fc.dt_contratacao,

    CAST(
        date_trunc('MONTH', fc.dt_contratacao) AS DATE
    ) AS safra_contratacao,

    fc.valor_contratado,
    fc.prazo_meses,

    dc.uf,
    dc.faixa_renda,
    dc.idade,
    dc.sexo,
    dc.estado_civil,

    dp.modalidade_credito,
    dp.taxa_juros,

    ds.faixa_score,
    ds.risco,

    COALESCE(ppc.qtd_pagamentos, 0) AS qtd_pagamentos,
    COALESCE(ppc.valor_total_pago, 0) AS valor_total_pago,
    ppc.media_dias_atraso,
    COALESCE(ppc.max_dias_atraso, 0) AS max_dias_atraso,
    COALESCE(ppc.qtd_pagamentos_atraso, 0) AS qtd_pagamentos_atraso,
    COALESCE(ppc.qtd_pagamentos_default_90, 0) AS qtd_pagamentos_default_90,
    COALESCE(ppc.valor_pago_em_atraso, 0) AS valor_pago_em_atraso,
    ppc.primeira_data_vencimento,
    ppc.ultima_data_pagamento,
    ppc.qtd_pagamentos_inconsistentes,

    CASE
        WHEN COALESCE(ppc.qtd_pagamentos_atraso, 0) > 0 THEN 1
        ELSE 0
    END AS flag_possui_atraso,

    CASE
        WHEN COALESCE(ppc.max_dias_atraso, 0) >= 90 THEN 1
        ELSE 0
    END AS flag_default_90,

    CASE
        WHEN COALESCE(ppc.max_dias_atraso, 0) >= 90 THEN fc.valor_contratado
        ELSE 0
    END AS valor_em_risco,

    CASE
        WHEN COALESCE(ppc.max_dias_atraso, 0) >= 90 THEN 'Default 90+'
        WHEN COALESCE(ppc.max_dias_atraso, 0) > 0 THEN 'Em atraso'
        WHEN COALESCE(ppc.qtd_pagamentos, 0) = 0 THEN 'Sem pagamento'
        ELSE 'Em dia'
    END AS status_contrato

FROM dev_credit_risk.silver.fact_contrato fc

LEFT JOIN pagamentos_por_contrato ppc
    ON fc.id_contrato = ppc.id_contrato

LEFT JOIN dev_credit_risk.silver.dim_cliente dc
    ON fc.id_cliente = dc.id_cliente

LEFT JOIN dev_credit_risk.silver.dim_produto_credito dp
    ON fc.id_produto = dp.id_produto

LEFT JOIN dev_credit_risk.silver.dim_score ds
    ON fc.id_score = ds.id_score;

In [0]:
%sql
 drop table dev_credit_risk.gold.fact_carteira_credito
--select * from dev_credit_risk.gold.fact_carteira_credito
--where qtd_pagamentos_inconsistentes > 0